In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 128
print("Device:", DEVICE)

In [ ]:
df = pd.read_csv(r"C:\Users\Ciel\mental-health-chatbot\data\raw\validation\master_validation_dataset_FIXED.csv")
df.shape

In [ ]:
STRESS_MODEL_PATH = r"C:\Users\Ciel\mental-health-chatbot\models\mental-roberta-v2\final"
EMOTION_MODEL_PATH = r"C:\Users\Ciel\mental-health-chatbot\models\mental-roberta-emotion5"

stress_tok = AutoTokenizer.from_pretrained(STRESS_MODEL_PATH)
stress_model = AutoModelForSequenceClassification.from_pretrained(STRESS_MODEL_PATH).to(DEVICE).eval()

emotion_tok = AutoTokenizer.from_pretrained(EMOTION_MODEL_PATH)
emotion_model = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL_PATH).to(DEVICE).eval()

EMOTION_TO_IDX = {'anxiety': 0, 'burnout': 1, 'sadness': 2, 'overwhelm': 3, 'neutral': 4}
IDX_TO_EMOTION = {v: k for k, v in EMOTION_TO_IDX.items()}

In [ ]:
@torch.no_grad()
def predict_stress(text):
    enc = stress_tok(text, max_length=MAX_LEN, truncation=True,
                      padding="max_length", return_tensors="pt").to(DEVICE)
    probs = F.softmax(stress_model(**enc).logits, dim=1).cpu().numpy()[0]
    stress_prob = float(probs[1])   # label 1 = stressed (подтверждено в Dreaddit)
    return ("Yes" if stress_prob >= 0.5 else "No"), stress_prob

@torch.no_grad()
def predict_emotion(text):
    enc = emotion_tok(text, max_length=MAX_LEN, truncation=True,
                       padding="max_length", return_tensors="pt").to(DEVICE)
    probs = F.softmax(emotion_model(**enc).logits, dim=1).cpu().numpy()[0]
    pred_idx = int(probs.argmax())
    return IDX_TO_EMOTION[pred_idx], probs

In [ ]:
stress_preds, stress_probs = [], []
emotion_preds, emotion_probs_all = [], []

for text in df["text"]:
    s_pred, s_prob = predict_stress(text)
    e_pred, e_probs = predict_emotion(text)
    stress_preds.append(s_pred)
    stress_probs.append(s_prob)
    emotion_preds.append(e_pred)
    emotion_probs_all.append(e_probs)

df["predicted_stress"] = stress_preds
df["predicted_stress_prob"] = stress_probs
df["predicted_emotion"] = emotion_preds
for i, cls in IDX_TO_EMOTION.items():
    df[f"{cls}_prob"] = [p[i] for p in emotion_probs_all]

df.head()

In [ ]:
df.to_csv("predictions_70.csv", index=False)
print("Saved:", df.shape)

In [ ]:
df_trunc = pd.read_csv(r"C:\Users\Ciel\mental-health-chatbot\data\raw\validation\texts_truncated_for_rerun.csv")

trunc_stress_preds, trunc_stress_probs = [], []
trunc_emotion_preds, trunc_emotion_probs_all = [], []

for text in df_trunc["text_truncated"]:
    s_pred, s_prob = predict_stress(text)
    e_pred, e_probs = predict_emotion(text)
    trunc_stress_preds.append(s_pred)
    trunc_stress_probs.append(s_prob)
    trunc_emotion_preds.append(e_pred)
    trunc_emotion_probs_all.append(e_probs)

df_trunc["predicted_stress_trunc"] = trunc_stress_preds
df_trunc["predicted_stress_prob_trunc"] = trunc_stress_probs
df_trunc["predicted_emotion_trunc"] = trunc_emotion_preds

df_trunc.to_csv("predictions_70_truncated.csv", index=False)
print("Saved:", df_trunc.shape)